# Manual annotation sets: baseline NREM, sleep deprivation, and two annotations of the last hour

Supplementary Tables S6b and S6c, the numbers quoted with them in the Supplementary Methods,
and the spike check of small OFF periods in early recovery NREM sleep. Early recovery NREM
itself (Supplementary Table S6a) comes from `batch_manual_vs_banded_and_morphological.ipynb`,
whose saved results this notebook reads.

- Baseline NREM (`Match`) and sleep deprivation (`SD`): the Table 1 engine,
  `head_to_head.head_to_head_experiment`, with the evaluation configs in
  `cnpix_local_sleep.evaluation.config`.
- The two wake annotations on the images both cover: `annotator_overlap.overlap_structure`.
- Spike check: `morphological.spike_validation.relaxed_off_spikes`.

The three runs re-detect banded OFFs per structure (long); each saves its results on NFS
next to Table 1's. The summary cells read the saved results.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon

from cnpix_local_sleep import sps_conf
from cnpix_local_sleep.evaluation import annotator_overlap, head_to_head
from cnpix_local_sleep.morphological import spike_validation
from cnpix_local_sleep.unit_based import banded

mua_source = "full48h"

# The two banded passes of Table 1 (batch_manual_vs_banded_and_morphological.ipynb).
BANDED_PASSES = {
    "banded-fixed_tiled": {
        "algo": "sticky",
        "band_definition": "fixed_tiled",
        "band_sizes": [250.0],
        "tile_start": "superficial",
        "param_strategy": "shared",
        "min_band_off_duration": 0.05,
        "min_merged_off_duration": 0.08,
    },
    "banded-greedy_fr": {
        "algo": "sticky",
        "band_definition": "greedy_fr",
        "param_strategy": "shared",
        "min_band_off_duration": 0.05,
        "min_merged_off_duration": 0.08,
    },
}
assert BANDED_PASSES["banded-fixed_tiled"] == banded.ROLLOUT_CONFIG

## Run baseline NREM and sleep deprivation

In [ ]:
for eval_name, mua_filters in (("Match", ("llas", "clas", "blas")), ("SD", ("llas", "clas"))):
    df = head_to_head.head_to_head_experiment(
        eval_name=eval_name,
        post_ms=(80,),
        scope="detection",
        banded_passes=BANDED_PASSES,
        mua_filters=mua_filters,
    )
    out = head_to_head.experiment_results_path(mua_source, eval_name)
    df.to_parquet(out)
    print("wrote", out, len(df), "rows")

## Run the two annotations of the last hour

In [ ]:
df = pd.concat(
    [
        annotator_overlap.overlap_structure(s, p, st, banded_passes=BANDED_PASSES)
        for s, p, st in annotator_overlap.overlap_structures()
    ],
    ignore_index=True,
)
df.to_parquet(annotator_overlap.results_path())
print("wrote", annotator_overlap.results_path(), len(df), "rows")

## Run the spike check (early recovery NREM)

In [ ]:
df = pd.concat(
    [
        spike_validation.relaxed_off_spikes(s, p, st)
        for s, p, st in head_to_head.labeled_cortical_structures("NREM")
    ],
    ignore_index=True,
)
df.to_parquet(spike_validation.results_path())
print("wrote", spike_validation.results_path(), len(df), "rows")

## Load

Rows are restricted to the annotated structures among the 29 subject-structure pairs analyzed
elsewhere, as in Table 1.

In [ ]:
K = ["subject", "probe", "structure"]
DETECTORS = {
    "llas": "Morphological, relaxed",
    "clas": "Morphological, stringent",
    "banded-fixed_tiled": "Unit-based, fixed-tiled",
    "banded-greedy_fr": "Unit-based, greedy-tiled",
}
ORDER = list(DETECTORS.values())
analysis_set = set(sps_conf.get_subject_probe_structure_list(method="morphological"))


def tidy(df):
    df = df[[k in analysis_set for k in zip(df["subject"], df["probe"], df["structure"])]]
    df = df[df["label"].isin(list(DETECTORS) + ["Wake", "SD"])].copy()
    df["detector"] = df["label"].map(DETECTORS).fillna("Other annotation")
    df["event_recall"] = df["event_sensitivity"]
    return df


SETS = {
    "Early recovery NREM": "NREM",
    "Circadian-matched baseline NREM": "Match",
    "Sleep deprivation, wake": "SD",
}
scores = {
    name: tidy(pd.read_parquet(head_to_head.experiment_results_path(mua_source, ev)))
    for name, ev in SETS.items()
}
overlap = tidy(pd.read_parquet(annotator_overlap.results_path()))
spikes = pd.read_parquet(spike_validation.results_path())
spikes = spikes[[k in analysis_set for k in zip(spikes["subject"], spikes["probe"], spikes["structure"])]]
for name, df in scores.items():
    print(f"{name}: {df[K].drop_duplicates().shape[0]} structures, {df['subject'].nunique()} rats")

## Supplementary Table S6b

In [ ]:
def mean_sd(x):
    return f"{x.mean():.2f} ± {x.std():.2f}"


rows = []
for metric in ("F1", "event_recall", "event_precision"):
    for det in ORDER:
        row = {"metric": metric, "detector": det}
        for name, df in scores.items():
            x = df.loc[df["detector"] == det, metric]
            n = df[K].drop_duplicates().shape[0]
            row[f"{name} ({n})"] = mean_sd(x) + (f" ({len(x)})" if len(x) < n else "")
        rows.append(row)
pd.DataFrame(rows)

## Greedy-tiled against the relaxed criterion on the same structures

Means over the structures where greedy-tiled bands could be constructed; Wilcoxon signed-rank
tests on the paired differences.

In [ ]:
for name, df in scores.items():
    wide = {
        m: df.pivot(index=K, columns="detector", values=m)
        for m in ("F1", "sensitivity", "precision", "event_recall")
    }
    pair = ["Unit-based, greedy-tiled", "Morphological, relaxed", "Morphological, stringent"]
    idx = wide["F1"][pair].dropna().index
    print(f"{name}, {len(idx)} structures")
    for m, w in wide.items():
        w = w.loc[idx]
        g, r = w["Unit-based, greedy-tiled"], w["Morphological, relaxed"]
        d = g - r
        print(
            f"  {m}: greedy {g.mean():.2f}, relaxed {r.mean():.2f}, "
            f"greedy higher {(d > 0).sum()}/{len(d)}, p {wilcoxon(d).pvalue:.2g}"
        )
    d = wide["F1"].loc[idx, "Unit-based, greedy-tiled"] - wide["F1"].loc[idx, "Morphological, stringent"]
    print(f"  F1 greedy - stringent: {d.mean():+.3f}, greedy higher {(d > 0).sum()}/{len(d)}, p {wilcoxon(d).pvalue:.2g}")

## F1 and the fraction of annotated pixels

Prevalence is the fraction of scored pixels inside an annotated OFF period. F1 at early
recovery prevalence recomputes each structure's F1 from its own sensitivity and
false-positive rate at the median prevalence of early recovery NREM.

In [ ]:
def f1_at(sens, fpr, prev):
    return 2 * sens * prev / (2 * sens * prev + fpr * (1 - prev) + (1 - sens) * prev)


relaxed = {}
for name, df in scores.items():
    x = df[df["detector"] == "Morphological, relaxed"].copy()
    x["prevalence"] = (x["TP"] + x["FN"]) / (x["TP"] + x["FP"] + x["FN"] + x["TN"])
    x["fpr"] = x["FP"] / (x["FP"] + x["TN"])
    relaxed[name] = x
p_nrem = relaxed["Early recovery NREM"]["prevalence"].median()
for name, x in relaxed.items():
    rho = {
        c: spearmanr(x[c], x["prevalence"]) for c in ("F1", "sensitivity", "fpr")
    }
    f1n = f1_at(x["sensitivity"], x["fpr"], p_nrem).mean()
    print(
        f"{name} (n {len(x)}): median prevalence {x['prevalence'].median():.4f}; "
        + ", ".join(f"rho({c}) {r[0]:+.2f} (p {r[1]:.2g})" for c, r in rho.items())
        + f"; F1 {x['F1'].mean():.2f}, at early recovery prevalence {f1n:.2f}"
    )

## Supplementary Table S6c

Every image both annotations cover. `Wake` is the last-hour annotation, `SD` the
sleep-deprivation annotation; each row scores the other annotation, or a detector, against the
reference.

In [ ]:
S6C_ORDER = ["Other annotation"] + ORDER
print(
    f"{overlap[K].drop_duplicates().shape[0]} structures in {overlap['subject'].nunique()} rats; "
    f"{overlap['minutes'].min():.0f}-{overlap['minutes'].max():.0f} min per structure"
)
rows = []
for det in S6C_ORDER:
    row = {"detector": det}
    for ref in ("Wake", "SD"):
        x = overlap[(overlap["reference"] == ref) & (overlap["detector"] == det)]
        for m in ("F1", "event_recall", "event_precision"):
            row[f"{ref}: {m}"] = mean_sd(x[m]) + (f" ({len(x)})" if len(x) < 13 else "")
    rows.append(row)
display(pd.DataFrame(rows))

for ref in ("Wake", "SD"):
    g = overlap[overlap["reference"] == ref]
    print(f"annotated per minute ({ref}): {(g['n_manual_events'] / g['minutes']).groupby([g[k] for k in K]).first().mean():.2f}")
g = overlap[overlap["reference"] == "Wake"]
for det in ORDER:
    x = g[g["detector"] == det]
    print(f"detected per minute, {det}: {(x['n_predicted_events'] / x['minutes']).mean():.1f}")
for ref in ("Wake", "SD"):
    w = overlap[overlap["reference"] == ref].pivot(index=K, columns="detector", values="F1")
    for base in ("Morphological, relaxed", "Other annotation"):
        for det in S6C_ORDER:
            if det == base:
                continue
            d = (w[det] - w[base]).dropna()
            print(
                f"  {ref}: F1 {det} - {base}: {d.mean():+.3f}, higher {(d > 0).sum()}/{len(d)}, "
                f"p {wilcoxon(d).pvalue:.2g}"
            )

## Spike check of small OFF periods (early recovery NREM)

Per structure, spikes inside the small OFF periods (relaxed but not stringent criterion) as a
fraction of the spikes in the same footprints at random times, separately for those that
matched a manual label and those that did not; medians across structures.

In [ ]:
small = spikes[spikes["small"]]
ratio = (
    small.groupby(K + ["matched"])[["obs", "exp_mean"]].sum()
    .eval("obs / exp_mean")
    .unstack("matched")
)
print(
    f"{len(ratio)} structures; small, unmatched: median {ratio[False].median():.2f}; "
    f"small, matched: median {ratio[True].median():.2f}"
)